# Local RAG Chatbot
## UChicago MS in Applied Data Science Assistant

In [1]:
# ============================================================
# PART 1 — MULTI-PAGE DATA COLLECTION & PREPROCESSING
# ============================================================


# ============================================================
# IMPORT LIBRARIES
# ============================================================

import requests
from bs4 import BeautifulSoup
import re
import json
import time
from collections import deque
from urllib.parse import urljoin, urlparse, urldefrag

from langchain_text_splitters import RecursiveCharacterTextSplitter


# ============================================================
# RECURSIVE CRAWL CONFIG
# ============================================================

START_URL = "https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/"

# Only follow pages under the MS ADS program section
ALLOWED_PATH_PREFIX = "/education/masters-programs/ms-in-applied-data-science"
ALLOWED_NETLOC = "datascience.uchicago.edu"

MAX_DEPTH = 10          # link hops from START_URL (0 = start page only)
MAX_PAGES = 40         # hard cap on pages scraped
REQUEST_TIMEOUT = 5   # seconds
REQUEST_DELAY = 0.3    # polite delay between requests (seconds)

UNWANTED_PHRASES = [
    "Skip to main content",
    "Learn More",
    "Apply Now",
    "Close Alert",
    "Menu",
    "Request Information",
]

SKIP_EXTENSIONS = (
    ".pdf", ".png", ".jpg", ".jpeg", ".gif", ".svg",
    ".zip", ".doc", ".docx", ".ppt", ".pptx", ".xls", ".xlsx",
)


# ============================================================
# CRAWL HELPERS
# ============================================================

def normalize_url(url):
    """Canonicalize URL for deduplication."""
    from urllib.parse import unquote

    url, _ = urldefrag(url.strip())
    parsed = urlparse(url)
    if parsed.scheme not in ("http", "https"):
        return None
    path = unquote(parsed.path or "/").strip()
    path = re.sub(r"\s+", "", path)  # fix hrefs like ".../online-program/ %20/"
    if not path.endswith("/"):
        path += "/"
    return f"{parsed.scheme}://{parsed.netloc}{path}"


def is_allowed_url(url):
    parsed = urlparse(url)
    if parsed.netloc != ALLOWED_NETLOC:
        return False
    if not parsed.path.startswith(ALLOWED_PATH_PREFIX):
        return False
    lower_path = parsed.path.lower()
    if any(lower_path.endswith(ext) for ext in SKIP_EXTENSIONS):
        return False
    return True


def extract_links(soup, base_url):
    links = set()
    for anchor in soup.find_all("a", href=True):
        href = anchor["href"].strip()
        if not href or href.startswith(("#", "mailto:", "tel:", "javascript:")):
            continue
        absolute = normalize_url(urljoin(base_url, href))
        if absolute and is_allowed_url(absolute):
            links.add(absolute)
    return links


def clean_page_text(raw_text):
    text = re.sub(r"\s+", " ", raw_text)
    for phrase in UNWANTED_PHRASES:
        text = text.replace(phrase, "")
    # Remove repeated site navigation menus that pollute every page
    text = re.sub(
        r"facet-arrow-down Overview In-Person Program.*?Career Outcomes Get In Touch",
        " ",
        text,
    )
    text = re.sub(r"MS in Applied Data Science Overview In-Person Program.*?Get In Touch", " ", text)
    text = re.sub(
        r"arrow-left-small|arrow-right-large|facet-arrow-down|link-out|mag-glass|Checked",
        " ",
        text,
    )
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def scrape_page_text(url):
    response = requests.get(url, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    for tag in soup([
        "script", "style", "nav", "footer", "header", "noscript", "iframe"
    ]):
        tag.extract()

    page_text = clean_page_text(soup.get_text(separator=" "))
    child_links = extract_links(soup, url)
    return page_text, child_links


def crawl_program_site(start_url, max_depth=MAX_DEPTH, max_pages=MAX_PAGES):
    start_url = normalize_url(start_url)
    queue = deque([(start_url, 0)])
    visited = set()
    scraped_urls = []
    pages = []  # list of {"url": ..., "text": ...}

    while queue and len(scraped_urls) < max_pages:
        url, depth = queue.popleft()
        if url in visited:
            continue
        visited.add(url)

        print(f"\nScraping (depth {depth}): {url}")
        try:
            page_text, child_links = scrape_page_text(url)
            print("Status: OK")
        except requests.RequestException as exc:
            print(f"Status: FAILED ({exc})")
            continue

        scraped_urls.append(url)
        if page_text:
            pages.append({"url": url, "text": page_text})

        if depth < max_depth:
            for link in sorted(child_links):
                if link not in visited:
                    queue.append((link, depth + 1))

        if REQUEST_DELAY:
            time.sleep(REQUEST_DELAY)

    return scraped_urls, pages


# ============================================================
# RECURSIVELY SCRAPE PROGRAM SUBLINKS
# ============================================================

scraped_urls, pages = crawl_program_site(START_URL)

all_text = "\n\n".join(page["text"] for page in pages)

print(f"\nALL PAGES SCRAPED SUCCESSFULLY")
print(f"Pages scraped: {len(scraped_urls)} (max_depth={MAX_DEPTH}, max_pages={MAX_PAGES})")

with open("ads_program_scraped_urls.json", "w", encoding="utf-8") as f:
    json.dump(scraped_urls, f, indent=2)

print("Scraped URL list saved to ads_program_scraped_urls.json")


# ============================================================
# SAVE CLEANED TEXT
# ============================================================

with open("ads_program_cleaned.txt", "w", encoding="utf-8") as f:
    f.write(all_text)

print("\nCombined cleaned text saved.")


# ============================================================
# CREATE TEXT SPLITTER
# ============================================================

splitter = RecursiveCharacterTextSplitter(

    chunk_size=900,
    chunk_overlap=150,

    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)


# ============================================================
# SPLIT INTO CHUNKS (deduplicated)
# ============================================================

chunk_records = []
for page in pages:
    for piece in splitter.split_text(page["text"]):
        piece = piece.strip()
        if piece:
            chunk_records.append({"text": piece, "source": page["url"]})

_seen_chunks = {}
for rec in chunk_records:
    if rec["text"] not in _seen_chunks:
        _seen_chunks[rec["text"]] = rec
chunk_records = list(_seen_chunks.values())
chunks = [rec["text"] for rec in chunk_records]

print("\nTotal Number of Chunks:", len(chunks))


# ============================================================
# DISPLAY SAMPLE CHUNKS
# ============================================================

print("\nFIRST CHUNK:\n")
print(chunks[0])

print("\nSECOND CHUNK:\n")
print(chunks[1])

print("\nTHIRD CHUNK:\n")
print(chunks[2])


# ============================================================
# SAVE CHUNKS
# ============================================================

with open("ads_program_chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunk_records, f, indent=2)

print("\nChunks saved successfully.")


# ============================================================
# CHUNK STATISTICS
# ============================================================

chunk_lengths = [len(chunk) for chunk in chunks]

print("\nAverage Chunk Length:", sum(chunk_lengths) / len(chunk_lengths))
print("Maximum Chunk Length:", max(chunk_lengths))
print("Minimum Chunk Length:", min(chunk_lengths))


# ============================================================
# SUCCESS MESSAGE
# ============================================================

print("""

PART 1 COMPLETED SUCCESSFULLY

Completed Tasks:
1. Recursive multi-page scraping (BFS with depth/page limits)
2. HTML parsing
3. Noise removal
4. Text cleaning
5. Semantic chunking
6. JSON export

Generated Files:
- ads_program_cleaned.txt
- ads_program_chunks.json
- ads_program_scraped_urls.json

""")

/Users/xcym/Desktop/Uchi_gen/Proj/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(



Scraping (depth 0): https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/
Status: OK

Scraping (depth 1): https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/course-progressions/
Status: OK

Scraping (depth 1): https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/faqs/
Status: OK

Scraping (depth 1): https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/in-person-program/
Status: OK

Scraping (depth 1): https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/online-program/
Status: OK

Scraping (depth 2): https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/capstone-project-archive/
Status: OK

Scraping (depth 2): https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/capstone-projects/
Status: OK

Scraping (depth 2): https://datascience.uchicago.edu/educa

In [2]:
# ============================================================
# PART 2 — OLLAMA EMBEDDING + VECTOR DATABASE
# ============================================================


# ============================================================
# IMPORT LIBRARIES
# ============================================================
import warnings
warnings.filterwarnings("ignore")
import os
import re
import shutil
import stat
import time

os.environ["ANONYMIZED_TELEMETRY"] = "False"

import json

from chromadb.config import Settings
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

RETRIEVAL_K = 8
RETRIEVAL_FETCH_K = 24
CHROMA_DIR = "./chroma_db"


def reset_chroma_directory(path):
    """Fully remove and recreate Chroma folder (avoids readonly DB errors)."""
    if "vector_db" in globals():
        del globals()["vector_db"]

    if os.path.isdir(path):
        def _on_rm_error(func, p, _exc):
            os.chmod(p, stat.S_IWUSR | stat.S_IREAD)
            func(p)
        shutil.rmtree(path, onerror=_on_rm_error)
        time.sleep(0.5)

    os.makedirs(path, mode=0o755, exist_ok=True)
# ============================================================
# LOAD CHUNKS
# ============================================================

with open("ads_program_chunks.json", "r", encoding="utf-8") as f:
    raw_chunks = json.load(f)

if raw_chunks and isinstance(raw_chunks[0], dict):
    chunks = [item["text"] for item in raw_chunks]
    metadatas = [{"source": item.get("source", "")} for item in raw_chunks]
else:
    chunks = raw_chunks
    metadatas = [{"source": ""} for _ in chunks]

print("Number of Loaded Chunks:", len(chunks))


# ============================================================
# INITIALIZE OLLAMA EMBEDDING MODEL
# ============================================================

embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)

print("Ollama embedding model initialized.")


# ============================================================
# CREATE CHROMA VECTOR DATABASE (fresh rebuild)
# ============================================================

reset_chroma_directory(CHROMA_DIR)
print("Reset chroma_db folder (fresh writable database).")

chroma_settings = Settings(
    anonymized_telemetry=False,
    allow_reset=True,
    is_persistent=True,
)

vector_db = Chroma.from_texts(
    texts=chunks,
    embedding=embedding_model,
    metadatas=metadatas,
    persist_directory=CHROMA_DIR,
    client_settings=chroma_settings,
    collection_name="ads_program",
)

print("Vector database created successfully.")
print("Chroma document count:", vector_db._collection.count())


# ============================================================
# RETRIEVAL HELPER (used in Parts 3 & 4)
# ============================================================

ADMISSION_BOOST_TERMS = (
    "application requirements",
    "how to apply",
    "candidate statement",
    "resume",
    "transcript",
    "recommendation",
    "programming supplement",
    "virtual portfolio",
    "bachelor",
    "letters of recommendation",
)

EXPANSION_PHRASES = (
    "expand",
    "more detail",
    "more details",
    "tell me more",
    "elaborate",
    "go on",
    "what else",
    "anything else",
    "explain more",
    "can you say more",
    "say more",
)


def doc_source(doc):
    return (doc.metadata or {}).get("source", "").lower()


def is_admission_question(question):
    q = question.lower()
    return any(
        term in q
        for term in (
            "admission",
            "apply",
            "application",
            "requirement",
            "eligible",
            "deadline",
            "transcript",
            "gre",
            "gmat",
            "toefl",
            "ielts",
        )
    )


def is_expansion_request(question):
    q = question.lower().strip()
    if q in {"more", "details", "more?", "details?"}:
        return True
    return any(phrase in q for phrase in EXPANSION_PHRASES)


def get_conversation_topic(history):
    """Last user question + start of last answer — anchors vague follow-ups."""
    last_user = ""
    last_assistant = ""
    for turn in reversed(history):
        if turn["role"] == "user" and not last_user:
            last_user = turn["content"]
        elif turn["role"] == "assistant" and not last_assistant:
            last_assistant = turn["content"][:300]
        if last_user and last_assistant:
            break
    return f"{last_user} {last_assistant}".strip()


def retrieve_context(question, k=RETRIEVAL_K, fetch_k=RETRIEVAL_FETCH_K):
    """Retrieve chunks with deduplication, admission-page boost, and re-ranking."""
    candidates = list(vector_db.similarity_search(question, k=fetch_k))

    if is_admission_question(question):
        admission_query = (
            "MS Applied Data Science application requirements transcripts resume "
            "candidate statement letters of recommendation programming supplement "
            "virtual portfolio how to apply"
        )
        candidates.extend(vector_db.similarity_search(admission_query, k=15))

    seen = set()
    unique = []
    for doc in candidates:
        key = doc.page_content.strip()
        if key and key not in seen:
            seen.add(key)
            unique.append(doc)

    query_words = {w.lower() for w in re.findall(r"\w+", question) if len(w) > 3}

    def relevance(doc):
        text = doc.page_content.lower()
        source = doc_source(doc)
        score = sum(1 for w in query_words if w in text)

        if is_admission_question(question):
            if "how-to-apply" in source:
                score += 6
            if "faqs" in source and "application" in text:
                score += 2
            score += sum(2 for term in ADMISSION_BOOST_TERMS if term in text)

        return score

    unique.sort(key=relevance, reverse=True)
    return unique[:k]


# ============================================================
# CHAT MEMORY + NATURAL ANSWERS (used in Parts 3 & 4)
# ============================================================

MAX_HISTORY_TURNS = 4


def format_chat_history(history):
    if not history:
        return "(No previous conversation.)"
    lines = []
    for turn in history[-(MAX_HISTORY_TURNS * 2):]:
        speaker = "User" if turn["role"] == "user" else "Assistant"
        lines.append(f"{speaker}: {turn['content']}")
    return "\n".join(lines)


def make_retrieval_query(question, history):
    """Build search query; handles follow-ups and 'expand more details'."""
    if is_expansion_request(question) and history:
        topic = get_conversation_topic(history)
        return f"{topic} detailed requirements information"

    if not history:
        return question

    recent_user = [
        turn["content"]
        for turn in history[-(MAX_HISTORY_TURNS * 2):]
        if turn["role"] == "user"
    ]
    return " ".join(recent_user[-2:] + [question])


def format_program_context(docs):
    return "\n\n".join(
        doc.page_content.strip() for doc in docs if doc.page_content.strip()
    )


def extract_verbatim_course_titles(context):
    titles = set()
    for match in re.finditer(
        r"(?:Core|Elective|Seminar|Foundational)\s+(.+?)(?:\s+Letter Grade|\s+Pass/Fail|\s+noncredit|:|\s+The |\s+You )",
        context,
        flags=re.IGNORECASE,
    ):
        title = match.group(1).strip()
        if 5 < len(title) < 90:
            titles.add(title)
    for match in re.finditer(
        r"(Machine Learning I{1,2}|Time Series Analysis and Forecasting|Career Seminar)",
        context,
        flags=re.IGNORECASE,
    ):
        titles.add(match.group(1).strip())
    return sorted(titles)


def build_answer_prompt(question, docs, history):
    context = format_program_context(docs)
    history_text = format_chat_history(history)
    course_titles = extract_verbatim_course_titles(context)
    course_hint = (
        ", ".join(course_titles)
        if course_titles
        else "(no specific course titles found in retrieved text)"
    )

    expansion_note = ""
    if is_expansion_request(question) and history:
        expansion_note = """
The user wants MORE DETAIL on the previous topic. Add new specifics from the program information.
Do not repeat your prior answer word-for-word. Use bullet points if helpful.
"""

    return f"""You are a helpful assistant for the University of Chicago MS in Applied Data Science program.

Answer using ONLY the program information below. Do not use outside knowledge.
{expansion_note}
Style (very important):
- Write naturally for a prospective student.
- NEVER mention passages, context, documents, retrieval, or numbered sources.
- NEVER say "based on the provided information", "the passages mention", "according to the program information", etc.
- Do not describe your sources; just state the facts.
- Use the recent conversation for follow-ups (e.g. "expand more details", "what about part-time?").
- If something is missing, say: "I don't see that detail on the program website."
- If you cannot answer at all, say: "I don't have that information on the program website."

Course names (anti-hallucination):
- ONLY name courses that appear verbatim in the program information OR in this allowed list: {course_hint}
- Never invent electives (e.g. do not make up "Financial Instruments and Markets" unless it appears above).
- For career advice, describe core/elective categories if specific titles are unavailable.

Program information:
{context}

Recent conversation:
{history_text}

Current question: {question}

Answer:"""


def answer_question(question, history, llm):
    search_query = make_retrieval_query(question, history)
    docs = retrieve_context(search_query)
    prompt = build_answer_prompt(question, docs, history)
    response = llm.invoke(prompt).strip()
    return response, docs


# ============================================================
# TEST RETRIEVAL
# ============================================================

query = "What are the admission requirements?"
results = retrieve_context(query)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print(f"\nTOP {len(results)} RETRIEVED CHUNKS:\n")

for i, result in enumerate(results):

    print(f"Result {i+1}:\n")

    print(result.page_content)

    print("\n")


# ============================================================
# SUCCESS MESSAGE
# ============================================================

print("""

PART 2 COMPLETED SUCCESSFULLY

Completed Tasks:
1. Loaded semantic chunks
2. Initialized Ollama embeddings
3. Generated vector embeddings
4. Created Chroma vector database
5. Performed semantic retrieval
6. Used similarity retrieval with deduplication and light re-ranking

Generated Files:
- chroma_db/

""")

Number of Loaded Chunks: 254
Ollama embedding model initialized.
Reset chroma_db folder (fresh writable database).
Vector database created successfully.
Chroma document count: 254

TOP 8 RETRIEVED CHUNKS:

Result 1:

. We do not accept letters of recommendation from family members, friends, or peers. Candidate Statement The candidate statement is a key part of your application. The admissions committee pays careful attention to how you understand and present your aims and qualifications. The application will include a prompt with detailed instructions on the information the program is looking for in your candidate statement. Best practices to keep in mind as you write: Do not restate your resume. Your statement should not exceed 700 words. Resume/CV The Master’s in Applied Data Science admissions committee will review your resume. We encourage you to take the time to carefully update the resume you submit to us


Result 2:

How to Apply | DSI MS in Applied Data Science Follow Master’s 

In [3]:
# ============================================================
# PART 3 — RAG ANSWER GENERATION
# ============================================================


# ============================================================
# IMPORT LIBRARIES
# ============================================================

from langchain_community.llms import Ollama


# ============================================================
# INITIALIZE LOCAL LLM
# ============================================================

llm = Ollama(
    model="llama3"
)

print("LLM initialized successfully.")


# ============================================================
# USER QUESTION
# ============================================================

question = "What are the admission requirements?"
chat_history = []  # empty for this single-shot test


# ============================================================
# GENERATE ANSWER (with chat-aware retrieval + natural prompt)
# ============================================================

response, retrieved_docs = answer_question(question, chat_history, llm)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("\nUSER QUESTION:\n")
print(question)

print("\nRETRIEVED CONTEXT (debug — not shown to the user in the final answer):\n")
print(format_program_context(retrieved_docs))

print("\nFINAL AI ANSWER:\n")
print(response)


# ============================================================
# SUCCESS MESSAGE
# ============================================================

print("""

PART 3 COMPLETED SUCCESSFULLY

Completed Tasks:
1. Retrieved relevant semantic chunks
2. Constructed RAG prompt
3. Sent augmented context to llama3
4. Generated grounded AI response
5. Reduced hallucination risk

System Pipeline:
User Question
↓
Semantic Retrieval
↓
Context Augmentation
↓
llama3 Generation
↓
Final AI Answer

""")

LLM initialized successfully.

USER QUESTION:

What are the admission requirements?

RETRIEVED CONTEXT (debug — not shown to the user in the final answer):

. We do not accept letters of recommendation from family members, friends, or peers. Candidate Statement The candidate statement is a key part of your application. The admissions committee pays careful attention to how you understand and present your aims and qualifications. The application will include a prompt with detailed instructions on the information the program is looking for in your candidate statement. Best practices to keep in mind as you write: Do not restate your resume. Your statement should not exceed 700 words. Resume/CV The Master’s in Applied Data Science admissions committee will review your resume. We encourage you to take the time to carefully update the resume you submit to us

How to Apply | DSI MS in Applied Data Science Follow Master’s in Applied Data Science Application Requirements The application portal 

In [5]:
# ============================================================
# PART 4 — INTERACTIVE RAG CHATBOT
# ============================================================


# ============================================================
# INTERACTIVE CHAT LOOP (with conversation memory)
# ============================================================

chat_history = []

print(
    "Tips: type 'exit' to quit, 'reset' to clear memory.\n"
    "Follow-ups work: e.g. ask about admissions, then 'expand more details'.\n"
)

while True:

    question = input("\nAsk a question about the UChicago ADS program: ").strip()

    if not question:
        continue

    if question.lower() == "exit":
        print("\nChatbot session ended.")
        break

    if question.lower() == "reset":
        chat_history = []
        print("\nConversation memory cleared.")
        continue

    response, retrieved_docs = answer_question(question, chat_history, llm)

    print("\n" + "=" * 60)
    print("\nUSER QUESTION:\n")
    print(question)
    print("\nRETRIEVED CONTEXT (debug):\n")
    print(format_program_context(retrieved_docs))
    print("\nFINAL AI ANSWER:\n")
    print(response)
    print("\n" + "=" * 60)

    chat_history.append({"role": "user", "content": question})
    chat_history.append({"role": "assistant", "content": response})


# ============================================================
# SUCCESS MESSAGE
# ============================================================

print("""

PART 4 COMPLETED SUCCESSFULLY

Completed Tasks:
1. Built interactive chatbot loop
2. Accepted dynamic user questions
3. Retrieved semantic context
4. Generated grounded AI responses
5. Enabled multi-turn chat with conversation memory

Final System:
User Question
↓
Semantic Retrieval
↓
Context Augmentation
↓
llama3 Generation
↓
Interactive AI Chatbot

""")

Tips: type 'exit' to quit, 'reset' to clear memory.
Follow-ups work: e.g. ask about admissions, then 'expand more details'.



USER QUESTION:

How long is the program?

RETRIEVED CONTEXT (debug):

Master's in Applied Data Science | DSI Elevate Your Expertise in Data Science The University of Chicago’s MS in Applied Data Science program equips you with in-demand expertise and an unparalleled network of global alumni. Take the next step and start your application today. How to Apply Programs Choose from full- and part-time options in our In-Person and Online programs. Apply today! In-Person Program The In-Person program, ideal for early career professionals, now includes a 2-year full-time option starting Autumn 2026—alongside the existing 12–15 month track for full- and part-time students. Online Program The same rigorous curriculum, faculty, and network as the in-person degree. Designed for working professionals with 2+ years of full-time experience. MBA/MS Program Differentiate yourse

can i graduate in one year?

what are courses i can take
what are some example course names


expand more details
tell me more
elaborate
what else?
more / details


thanks